# 03 Bronze trades
Ingests trades.csv from the landing Volume into bronze.trades using Auto Loader.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS bronze.checkpoints

In [0]:
from pyspark.sql import functions as F

source = "/Volumes/workspace/bronze/landing/"
chk = "/Volumes/workspace/bronze/checkpoints/trades"

stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", True)
    .option("pathGlobFilter", "trades*.csv")
    .option("cloudFiles.schemaLocation", chk + "/schema")
    .load(source)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    stream.writeStream
    .option("checkpointLocation", chk + "/state")
    .trigger(availableNow=True)
    .toTable("bronze.trades")
)

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM bronze.trades

In [0]:
%sql
SELECT * FROM bronze.trades LIMIT 10